# Thinking-mode diagnostic — no training

Can thinking mode help the same model on arithmetic and the remaining development errors? Evaluate ten arithmetic questions on the base and adapter, plus all 84 F/R development questions on the adapter. Keep a greedy reproduction arm, then compare sampled non-thinking and sampled thinking with identical sampling settings and a 1,024-token cap: **312 answers, zero training updates**.

This notebook saves generated visible reasoning text and scores the final answer separately. An unfinished thinking block cannot count as a correct answer. Visible text is not proof of internal reasoning, and M1 traces are not post-unlearning leakage results. Original pilot gates/results remain unchanged.

Use the existing Kaggle GPU runtime with HF_TOKEN attached. **Publish the new code, config and verified utility-run reference files to GitHub main before executing this notebook.**


In [1]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle GPU kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='thinking-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('src/evaluation/mode_diagnostic.py').is_file(), 'Publish the new thinking-mode diagnostic files before running this notebook.'
assert Path('experiments/runs/m1_utility_prompt_diagnostic_20261105/base_arithmetic-v1.jsonl').is_file(), 'Publish the verified utility reference records too.'


Cloning into '/kaggle/working/thinking-source-4qboc6jw/repo'...


Repository commit: 3c77324a4c70f987074786ec3810230a3fb53bac


In [2]:
assert Path('/kaggle/working').is_dir()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__)'], check=True)


2.11.0+cu130 5.20.2 0.21.2


CompletedProcess(args=['/usr/bin/python3', '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__)'], returncode=0)

In [3]:
import json

plan_dir = Path(tempfile.mkdtemp(prefix='thinking-plan-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.evaluation.mode_diagnostic', '--config', 'configs/evaluation/thinking_mode_diagnostic_v1.json', '--output-dir', str(plan_dir), '--mode', 'plan'], check=True)
plan = json.loads((plan_dir / 'plan.json').read_text())
assert plan['prediction_count'] == 312 and plan['training_updates'] == 0
print('Reference hashes passed. Planned answers:', plan['prediction_count'])


{
  "config": {
    "run_type": "thinking_mode_diagnostic_v1",
    "model_id": "Qwen/Qwen3-0.6B",
    "model_revision": "c1899de289a04d12100db370d81485cdf75e47ca",
    "seed": 42,
    "batch_size": 16,
    "sampled_max_new_tokens": 1024,
    "sampling": {
      "do_sample": true,
      "temperature": 0.6,
      "top_p": 0.95,
      "top_k": 20,
      "min_p": 0.0
    },
    "artifact": {
      "repo_id": "tharundevulapalli/persistent-llm-unlearning-checkpoints",
      "revision": "07d23d8b904ab48401f9c6676d05bb85a14d9fbd",
      "path": "diagnostics/m1-tiny/m1-tiny-run-8myx_22r"
    },
    "reference_dir": "experiments/runs/m1_tiny_attempt_20261105",
    "reference_sha256": {
      "summary.json": "f49264fd0814c9cbea2843f6df9f69168eb8d752910c2902e9b9794cb8f0344a",
      "selected_dev.jsonl": "47e18fc280aa1c81cbaed4dc5677ba5f2dcc4fe0cef8c484d2b8685086b64f4b",
      "post_dev.jsonl": "243b73a2bdbbb865661ab8dc7885833b44c7f05bd52c7746219d3c6e9d53da52",
      "persistence.json": "587e1a339b

In [4]:
from kaggle_secrets import UserSecretsClient

evaluation_env = os.environ.copy()
evaluation_env['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
output_dir = Path(tempfile.mkdtemp(prefix='m1-thinking-mode-', dir=remote_root))
print('Evaluation outputs:', output_dir, flush=True)
try:
    subprocess.run([sys.executable, '-u', '-m', 'src.evaluation.mode_diagnostic', '--config', 'configs/evaluation/thinking_mode_diagnostic_v1.json', '--output-dir', str(output_dir), '--mode', 'evaluate'], env=evaluation_env, check=True)
finally:
    evaluation_env.pop('HF_TOKEN', None)


Evaluation outputs: /kaggle/working/m1-thinking-mode-ua9h7f7s


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 452.72it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


base_utility_greedy_reference: 7/10 strict correct; 0 caps; 0 boundary failures
Thinking=False: generated 10/10 predictions
base_utility_sampled_no_thinking: 7/10 strict correct; 0 caps; 0 boundary failures
Thinking=True: generated 10/10 predictions
base_utility_sampled_thinking: 10/10 strict correct; 0 caps; 0 boundary failures
adapter_utility_greedy_reference: 6/10 strict correct; 0 caps; 0 boundary failures
Thinking=False: generated 10/10 predictions
adapter_utility_sampled_no_thinking: 6/10 strict correct; 0 caps; 0 boundary failures
Thinking=True: generated 10/10 predictions
adapter_utility_sampled_thinking: 10/10 strict correct; 0 caps; 0 boundary failures
adapter_development_greedy_reference: 73/84 strict correct; 0 caps; 0 boundary failures
Thinking=False: generated 16/84 predictions
Thinking=False: generated 32/84 predictions
Thinking=False: generated 48/84 predictions
Thinking=False: generated 64/84 predictions
Thinking=False: generated 80/84 predictions
Thinking=False: gener

In [5]:
summary = json.loads((output_dir / 'summary.json').read_text())
assert summary['execution_complete'] and summary['training_updates'] == 0
print('Greedy controls reproduced:', summary['greedy_reproduction'])
for name, group in sorted(summary['groups'].items()):
    print(name, '| strict:', str(group['answer_matches']) + '/' + str(group['count']), '| caps:', group['token_limit_hits'], '| missing thinking boundaries:', group['missing_thinking_boundaries'])
    if 'learning_metrics' in group:
        print('Development groups:', json.dumps(group['learning_metrics']))
for checkpoint in ['base', 'adapter']:
    file = output_dir / (checkpoint + '_utility_sampled_thinking.jsonl')
    rows = [json.loads(line) for line in file.read_text().splitlines()]
    print('\n', checkpoint, 'example visible reasoning:', rows[0]['visible_reasoning'][:1500])
    print('Example final answer:', rows[0]['final_answer_text'])
    for row in rows:
        print(row['example_id'], '| expected:', row['answer'], '| final:', repr(row['final_answer_text']), '| correct:', row['answer_match'])
print('One sampling seed and bounded outputs: inspect caps/boundaries and raw evidence before drawing conclusions. Original pilot remains failed.')


Greedy controls reproduced: {'adapter/development': {'different_example_ids': [], 'raw_predictions_identical': True}, 'adapter/utility': {'different_example_ids': [], 'raw_predictions_identical': True}, 'base/utility': {'different_example_ids': [], 'raw_predictions_identical': True}}
adapter_development_greedy_reference | strict: 73/84 | caps: 0 | missing thinking boundaries: 0
Development groups: {"F/author_book": {"accuracy": 1.0, "correct": 8, "count": 8}, "F/multi_hop": {"accuracy": 0.9166666666666666, "correct": 11, "count": 12}, "R/author_book": {"accuracy": 1.0, "correct": 16, "count": 16}, "R/multi_hop": {"accuracy": 0.6666666666666666, "correct": 16, "count": 24}}
adapter_development_sampled_no_thinking | strict: 73/84 | caps: 0 | missing thinking boundaries: 0
Development groups: {"F/author_book": {"accuracy": 1.0, "correct": 8, "count": 8}, "F/multi_hop": {"accuracy": 0.9166666666666666, "correct": 11, "count": 12}, "R/author_book": {"accuracy": 1.0, "correct": 16, "count": 

In [6]:
import zipfile

archive = Path(str(output_dir) + '-evidence.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Download from Kaggle Output:', archive)


Download from Kaggle Output: /kaggle/working/m1-thinking-mode-ua9h7f7s-evidence.zip
